# MLP Gate Analysis

Analyze MLP activation patterns: distribution, pre/post relationship,
neuron frequency, output direction, and MLP vs attention balance.

## Why This Matters

MLP gate provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_gate_analysis import (
    mlp_activation_distribution, mlp_pre_post_relationship,
    neuron_activation_frequency, mlp_output_direction_analysis,
    mlp_contribution_vs_attention,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Activation Distribution

In [ ]:
result = mlp_activation_distribution(model, tokens, layer=0)
print(f"Active fraction: {result['active_fraction']:.2%}")
print(f"Dead neurons: {result['n_dead_neurons']} ({result['dead_fraction']:.1%})")
print(f"Mean: {result['mean_activation']:.4f}, Std: {result['std_activation']:.4f}")
print(f"Range: [{result['min_activation']:.4f}, {result['max_activation']:.4f}]")

## Pre/Post Relationship

In [ ]:
result = mlp_pre_post_relationship(model, tokens, layer=0)
print(f"Correlation: {result['correlation']:.4f}")
print(f"Mean ratio: {result['mean_ratio']:.4f}")
print(f"Pass rate: {result['pass_rate']:.2%}")
print(f"Suppression: {result['suppression_rate']:.2%}")

## Neuron Activation Frequency

In [ ]:
result = neuron_activation_frequency(model, tokens, layer=0, top_k=10)
print(f"Mean frequency: {result['mean_frequency']:.2%}")
print(f"Always active: {result['n_always_active']}, Never active: {result['n_never_active']}\n")
print('Most active:')
for n in result['most_active'][:5]:
    print(f"  Neuron {n['neuron_idx']:3d}: freq={n['frequency']:.2%}, mean_act={n['mean_activation']:.4f}")
if result['least_active']:
    print('\nLeast active:')
    for n in result['least_active'][:5]:
        print(f"  Neuron {n['neuron_idx']:3d}: freq={n['frequency']:.2%}, mean_act={n['mean_activation']:.4f}")

## Output Direction Analysis

In [ ]:
result = mlp_output_direction_analysis(model, tokens, layer=0)
cons = 'CONSISTENT' if result['is_consistent'] else 'variable'
print(f"Mean norm: {result['mean_norm']:.4f}, alignment: {result['mean_alignment']:.4f} [{cons}]\n")
for p in result['per_position']:
    print(f"  Pos {p['position']}: norm={p['norm']:.4f}, "
          f"align={p['alignment_to_mean']:+.4f}")

## MLP vs Attention Balance

In [ ]:
result = mlp_contribution_vs_attention(model, tokens)
print(f"MLP dominant: {result['n_mlp_dominant']}, Attn dominant: {result['n_attn_dominant']}\n")
for p in result['per_layer']:
    dom = 'MLP' if p['mlp_fraction'] > 0.5 else 'attn'
    coop = '+' if p['cooperate'] else '-'
    print(f"  Layer {p['layer']}: attn={p['attn_norm']:.4f}, "
          f"mlp={p['mlp_norm']:.4f} ({p['mlp_fraction']:.1%} MLP) "
          f"[{dom}] align={p['alignment']:+.4f} [{coop}]")